# Baseline 3D-patch — W2 ngày 5

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

Train DenseNet121-3D (8 pha ghép kênh → 7 lớp) trên **cache đã tiền xử lý**, một fold,
để lấy **số mốc đầu tiên** của dự án. Chưa cần CI — bootstrap CI là việc W3.

**Thứ tự:**
1. Bootstrap (clone code + tìm cache đã mount)
2. Smoke test: dataset + model forward (rẻ, chạy trước khi tốn GPU)
3. Train fold 1
4. Đọc số + ma trận nhầm lẫn

⚠️ Notebook **không được chạm test-104** (AGENTS.md §3.4). Ở đây chỉ có train/val fold.

⚠️ Kaggle cắt session bất cứ lúc nào. Chạy lại đúng cell train sẽ **resume** từ `last.pt` —
không mất tiến trình, không train lại từ đầu.

## 0. Bootstrap

Luôn xoá + clone lại. Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

Cache lấy từ Kaggle Dataset `marcohoang/lld-mmri-3` (Private, xem `configs/preprocess.yaml`).
Không đoán đường dẫn mount — tìm thư mục nào chứa `cache_meta.json`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"
ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    subprocess.run(["rm", "-rf", str(REPO)], check=False)
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
    sys.path.insert(0, str(REPO))
    os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/baseline_3dpatch"
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(REPO))

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())


def find_cache_dir(search_root="/kaggle/input", max_depth=3):
    """Tìm thư mục cache bằng chính file metadata của nó, không đoán sơ đồ mount."""
    base = Path(search_root)
    if not base.is_dir():
        return None
    for depth in range(max_depth + 1):
        pattern = "/".join(["*"] * depth + ["cache_meta.json"])
        for hit in sorted(base.glob(pattern)):
            return hit.parent
    return None


if ON_KAGGLE:
    cache_dir = find_cache_dir()
    if cache_dir is None:
        raise SystemExit(
            "Không thấy cache. Add data -> Datasets -> marcohoang/lld-mmri-3 (version 1)."
        )
    os.environ["LLDMMRI_CACHE_DIR"] = str(cache_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "monai"], check=False)

from src.utils.io import load_yaml, resolve_cache_dir  # noqa: E402

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
CACHE_DIR = resolve_cache_dir(CFG)

print("cache dir :", CACHE_DIR)
print("file .npz :", len(list(CACHE_DIR.glob("*.npz"))), "(cần 498)")
print("meta      :", (CACHE_DIR / "cache_meta.json").read_text(encoding="utf-8")[:400])

## 1. Smoke test — dataset + model

Rẻ và chạy trước khi tốn GPU: batch phải ra đúng `[B, 8, 96, 96, 48]`, model phải
trả logits `[B, 7]`. Cũng in phân bố lớp của fold — để biết lớp hiếm hiếm tới mức nào.

In [ ]:
from collections import Counter

import torch
from torch.utils.data import DataLoader

from src.data.dataset import build_fold_datasets
from src.data.taxonomy import SHORT_NAMES
from src.models import build_model, count_parameters
from src.train.loop import class_weights_from_labels

FOLD = CFG["fold"]
train_ds, val_ds = build_fold_datasets(CACHE_DIR, FOLD, splits_dir=REPO / "splits")
print(f"fold {FOLD}: train={len(train_ds)} val={len(val_ds)}")

train_labels = [label for _, label, _ in train_ds.samples]
counts = Counter(train_labels)
weights = class_weights_from_labels(train_labels)
for k in sorted(SHORT_NAMES):
    print(f"  {SHORT_NAMES[k]:>7}: {counts.get(k, 0):3d} ca | trọng số {weights[k]:.2f}")

batch = next(iter(DataLoader(train_ds, batch_size=2, shuffle=False)))
model = build_model(CFG["model"])
model.eval()
with torch.no_grad():
    logits = model(batch["image"])

print("batch image:", tuple(batch["image"].shape), "| logits:", tuple(logits.shape))
print("tham số    :", f"{count_parameters(model):,}")
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Train fold 1

Toàn bộ hyperparam nằm trong `configs/baseline_3dpatch.yaml` — sửa ở đó rồi commit,
**đừng sửa trong notebook** (số báo cáo phải tái lập được từ config + seed).

Nếu session chết giữa chừng: chạy lại đúng cell này, nó tự tiếp từ epoch dở dang.

In [ ]:
from src.train.run import train

result = train(CFG_PATH, fold_override=FOLD)
print(result)

## 3. Số mốc + ma trận nhầm lẫn

Đây là **số val của 1 fold, 1 seed** — mốc để so, không phải kết quả báo cáo.
Kết quả báo cáo cần CV 5-fold + bootstrap CI (W3, AGENTS.md §3.5).

In [ ]:
import json

import numpy as np

from src.utils.io import resolve_output_dir

run_dir = resolve_output_dir(CFG) / f"fold{FOLD}"
best = json.loads((run_dir / "metrics_best.json").read_text(encoding="utf-8"))

print(f"fold {best['fold']} · epoch {best['epoch']} · seed {best['seed']}")
for key in ("macro_f1", "balanced_accuracy", "accuracy", "cohen_kappa"):
    print(f"  {key:>18}: {best[key]:.4f}")

print("\nF1 từng lớp:")
for k, f1 in enumerate(best["per_class_f1"]):
    print(f"  {SHORT_NAMES[k]:>7}: {f1:.3f}")

print("\nMa trận nhầm lẫn (hàng = thật, cột = đoán):")
matrix = np.array(best["confusion_matrix"])
header = "        " + "".join(f"{SHORT_NAMES[k]:>8}" for k in sorted(SHORT_NAMES))
print(header)
for k, row in enumerate(matrix):
    print(f"{SHORT_NAMES[k]:>7} " + "".join(f"{v:>8d}" for v in row))

print("\n--- train_log.csv (10 epoch cuối) ---")
print("".join((run_dir / "train_log.csv").read_text(encoding="utf-8").splitlines(True)[-10:]))

## 4. Giữ lại gì

`/kaggle/working/runs/...` biến mất khi session kết thúc → **tải về hoặc save output**:

- `metrics_best.json`, `train_log.csv`, `config_used.json` → chép số vào WORKLOG.
- `val_probs_best.npz` → W3 tính bootstrap CI, W5 tính calibration/selective **mà không train lại**.
- `best.pt`, `last.pt` → checkpoint. **Không commit vào git** (AGENTS.md §3.10).